# Results Dashboard

Load experiment results from a completed CLARYON run and visualize metrics, predictions, and model comparisons.

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.datasets import load_iris

# Run a quick experiment to generate results
iris = load_iris()
df = pd.DataFrame(iris.data, columns=[f"f{i}" for i in range(4)])
df["label"] = (iris.target > 0).astype(int)
df.insert(0, "Key", [f"S{i:04d}" for i in range(len(df))])

tmp = tempfile.mkdtemp()
csv_path = Path(tmp) / "data.csv"
df.to_csv(csv_path, sep=";", index=False)

import yaml
config_dict = {
    "experiment": {"name": "dashboard_demo", "seed": 42, "results_dir": str(Path(tmp) / "Results")},
    "data": {"tabular": {"path": str(csv_path), "label_col": "label", "id_col": "Key", "sep": ";"}},
    "cv": {"strategy": "kfold", "n_folds": 5, "seeds": [42]},
    "models": [{"name": "mlp", "type": "tabular"}],
    "evaluation": {"metrics": ["bacc", "auc", "sensitivity", "specificity"]},
    "reporting": {"markdown": True},
}

config_path = Path(tmp) / "config.yaml"
with open(config_path, "w") as f:
    yaml.dump(config_dict, f)

import logging
logging.basicConfig(level=logging.WARNING)
from claryon.config_schema import load_config
from claryon.pipeline import run_pipeline
state = run_pipeline(load_config(config_path))
print("Experiment complete!")

In [ ]:
# Load metrics summary
results_dir = Path(tmp) / "Results"
summary = pd.read_csv(results_dir / "metrics_summary.csv", sep=";")
display(summary)

In [ ]:
# Aggregate predictions across folds
from claryon.io.predictions import read_predictions

all_preds = []
for fold_dir in sorted((results_dir / "mlp" / "seed_42").glob("fold_*")):
    pred_file = fold_dir / "Predictions.csv"
    if pred_file.exists():
        fold_df = read_predictions(pred_file)
        all_preds.append(fold_df)

combined = pd.concat(all_preds, ignore_index=True)
print(f"Total predictions: {len(combined)}")
print(f"\nPer-fold accuracy:")
for fold, grp in combined.groupby("Fold"):
    acc = (grp["Actual"] == grp["Predicted"]).mean()
    print(f"  Fold {fold}: {acc:.3f}")

In [ ]:
# Confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(combined["Actual"], combined["Predicted"])
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=["Class 0", "Class 1"]).plot(ax=ax)
ax.set_title("MLP — Aggregated Confusion Matrix")
plt.tight_layout()
plt.show()